<img src="https://devra.ai/analyst/notebook/4516/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
    <div style="font-size:150%; color:#FEE100"><b>Supply Chain Analysis and Revenue Predictor</b></div>
    <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

It is fascinating to observe how supply chain intricacies can be leveraged to not only gain operational insights but also predict revenue outcomes. If you find these insights useful, please consider upvoting.

## Table of Contents

- [Load Data](#Load-Data)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Prediction Model: Revenue Generation](#Prediction-Model:-Revenue-Generation)
- [Conclusion and Future Work](#Conclusion-and-Future-Work)

In [ ]:
# Import necessary libraries and suppress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # use Agg backend for matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

# Set plot style
sns.set(style='whitegrid')

## Load Data

Let's load the supply chain dataset which provides detailed insights into supply chain parameters including product details, sales, shipping information, production costs, and more.

In [ ]:
# Load the dataset
file_path = '/kaggle/input/supply-chain-analysis-dataset/supply_chain_data.csv'
df = pd.read_csv(file_path, encoding='ascii', delimiter=',')

# Display basic dataset information
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())

## Data Cleaning and Preprocessing

In this section we check for missing values, duplicates, and correct data types. Note that although no date columns are explicitly provided, methods to infer data types are important for any dataset. We also convert columns to their respective numeric types where applicable.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print('Missing values in each column:')
print(missing_values)

# Check for duplicate rows
duplicates = df.duplicated().sum()
print('\nDuplicate rows:', duplicates)

# Convert columns assumed to be numeric (if necessary)
numeric_cols = ['Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Stock levels', 
                'Lead times', 'Order quantities', 'Shipping times', 'Shipping costs', 'Lead time', 
                'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Defect rates', 'Costs']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with any missing numeric data if they exist
df.dropna(inplace=True)

# Reset index after cleaning
df.reset_index(drop=True, inplace=True)

print('\nAfter cleaning, dataset shape:', df.shape)

## Exploratory Data Analysis

In this section we generate multiple visualizations to help us understand the distribution and relationships within the data. We include histograms, pair plots, box plots and a correlation heatmap for numeric features. Note that correlation heatmaps are only presented if there are at least four numeric columns.

In [ ]:
# Plot histogram for Price distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Price'], kde=True, color='skyblue')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Pair plot for selected numeric columns (if dataset has enough numeric vars)
selected_cols = ['Price', 'Availability', 'Number of products sold', 'Revenue generated']
sns.pairplot(df[selected_cols])
plt.show()

# Box plot for Shipping costs
plt.figure(figsize=(6, 4))
sns.boxplot(x=df['Shipping costs'], color='lightgreen')
plt.title('Box Plot of Shipping Costs')
plt.xlabel('Shipping costs')
plt.tight_layout()
plt.show()

# Correlation heatmap for numeric features if there are 4 or more numeric columns
numeric_df = df.select_dtypes(include=[np.number])
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(12, 10))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Heatmap for Numeric Features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for a correlation heatmap.')

# Count plot for Product type
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Product type', palette='pastel')
plt.title('Count Plot of Product Types')
plt.xlabel('Product type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Prediction Model: Revenue Generation

Given the data, it is reasonable to predict the revenue generated based on various numeric supply chain attributes. We use a simple linear regression model as a predictor. The model is trained on a subset of features and its performance is evaluated using the R² score. Finally, we apply permutation importance to understand which features most influence revenue predictions.

In [ ]:
# Define the target variable and features for the model
target = 'Revenue generated'

# We select numeric columns excluding the target as predictors
numeric_features = numeric_df.columns.tolist()
if target in numeric_features:
    numeric_features.remove(target)

# To avoid multicollinearity issues or overfitting, here we simply use all available numeric features.
X = df[numeric_features]
y = df[target]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on the test set
y_pred = lr_model.predict(X_test)

# Evaluate model performance using R² score
score = r2_score(y_test, y_pred)
print('Linear Regression R² score:', score)

# Compute permutation importance
perm_importance = permutation_importance(lr_model, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.barh(np.array(numeric_features)[sorted_idx], perm_importance.importances_mean[sorted_idx], color='coral')
plt.xlabel('Permutation Importance')
plt.title('Permutation Importance of Features')
plt.tight_layout()
plt.show()

## Conclusion and Future Work

This notebook presented an exploration of a supply chain dataset, from data loading and cleaning to detailed exploratory data analysis using multiple visualization techniques. We then built a linear regression predictor to estimate revenue generation, with the model evaluated via the R² score and further explained by permutation importance analysis.

The approach showcased here has the merit of blending both domain insights from EDA and a simple yet effective predictive model. In the future, it would be interesting to explore more sophisticated models, e.g., ensemble methods, and incorporate categorical feature encoding techniques to further boost predictive performance. Additionally, time-based analyses or forecasting models could be developed if temporal data were available.

If you found this notebook engaging and useful, please consider upvoting it.